# Text/Information Extraction Evaluation

Evaluating NER, relation extraction, schema conformity, and table extraction: SpaCy NER, entity level metrics, edit distance, LLM based extraction, Pydantic/JSON schema validation, and structured data extraction.

In [1]:
# !pip install spacy Levenshtein jsonschema pydantic scikit-learn

In [ ]:
GROQ_MODEL_FAST  = "llama-3.1-8b-instant"
GROQ_MODEL_SMART = "llama-3.3-70b-versatile"
OPENAI_BASE_URL = "https://api.groq.com/openai/v1"

import os
import json, re
import numpy as np
from groq import Groq
from dotenv import load_dotenv
load_dotenv()

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def groq_chat(prompt, system="You are a helpful assistant.",
              model=GROQ_MODEL_SMART, temperature=0.0, max_tokens=600):
    resp = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system",  "content": system},
            {"role": "user",    "content": prompt}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return resp.choices[0].message.content.strip()

def parse_json(raw):
    raw = re.sub(r'```json|```', '', raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        m = re.search(r'\{.*\}', raw, re.DOTALL)
        return json.loads(m.group()) if m else {}

def show(title, d):
    print(f"\n{'='*60}\n  📊 {title}\n{'='*60}")
    for k, v in d.items():
        print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

print('🤖 Groq test:', groq_chat('Reply with only: Groq is ready!', max_tokens=20))

✅ All packages installed!
🤖 Groq test: Groq is ready!


## 1. SpaCy NER (Named Entity Recognition)
### Tool: spaCy en_core_web_sm model

In [2]:
# !python -m spacy download en_core_web_sm

In [4]:
import spacy
nlp = spacy.load('en_core_web_sm')

text = ('Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976. '
        'Elon Musk, CEO of Tesla, met with Barack Obama in Washington D.C. '
        'Microsoft acquired LinkedIn for $26 billion in 2016.')

doc = nlp(text)
print('\n🏷️  spaCy NER Entities')
print('='*60)
for ent in doc.ents:
    print(f'  {ent.text:<25}  ->  {ent.label_:<10}  ({spacy.explain(ent.label_)})')


🏷️  spaCy NER Entities
  Apple Inc.                 ->  ORG         (Companies, agencies, institutions, etc.)
  Steve Jobs                 ->  PERSON      (People, including fictional)
  Cupertino                  ->  GPE         (Countries, cities, states)
  California                 ->  GPE         (Countries, cities, states)
  1976                       ->  DATE        (Absolute or relative dates or periods)
  Elon Musk                  ->  PERSON      (People, including fictional)
  Tesla                      ->  ORG         (Companies, agencies, institutions, etc.)
  Barack Obama               ->  PERSON      (People, including fictional)
  Washington D.C.            ->  ORG         (Companies, agencies, institutions, etc.)
  Microsoft                  ->  ORG         (Companies, agencies, institutions, etc.)
  LinkedIn                   ->  ORG         (Companies, agencies, institutions, etc.)
  $26 billion                ->  MONEY       (Monetary values, including unit)
  2016

## 2. NER Evaluation Metrics (Precision, Recall, F1, Exact/Partial Match)
### Tool: sklearn + custom metrics

In [5]:
# ── 2a. Entity-level Precision, Recall, F1 ────────────────
gold_entities = [
    ('Apple Inc.',     'ORG'), ('Steve Jobs',    'PERSON'), ('Cupertino',     'GPE'),
    ('California',     'GPE'), ('Elon Musk',     'PERSON'), ('Tesla',         'ORG'),
    ('Barack Obama',   'PERSON'), ('Washington D.C.', 'GPE'),
    ('Microsoft',      'ORG'), ('LinkedIn',      'ORG'),
]
predicted_entities = [(e.text, e.label_) for e in doc.ents]

def ner_metrics(gold, pred):
    g, p = set(gold), set(pred)
    tp = len(g & p); fp = len(p - g); fn = len(g - p)
    pr = tp/(tp+fp) if tp+fp else 0.0
    rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    return {'precision': pr, 'recall': rc, 'f1': f1, 'TP': tp, 'FP': fp, 'FN': fn}

show('NER — Precision / Recall / F1', ner_metrics(gold_entities, predicted_entities))

# Show per-entity comparison
print("\n  Detailed Entity Comparison:")
gold_set = set(gold_entities)
pred_set = set(predicted_entities)
for e in sorted(gold_set | pred_set, key=lambda x: x[0]):
    in_gold = e in gold_set
    in_pred = e in pred_set
    if in_gold and in_pred:
        print(f"    ✅ TP  {e[0]:<25} ({e[1]})")
    elif in_pred and not in_gold:
        print(f"    🔴 FP  {e[0]:<25} ({e[1]})")
    else:
        print(f"    ⚪ FN  {e[0]:<25} ({e[1]})")


  📊 NER — Precision / Recall / F1
  precision                     : 0.6923
  recall                        : 0.9000
  f1                            : 0.7826
  TP                            : 9
  FP                            : 4
  FN                            : 1

  Detailed Entity Comparison:
    🔴 FP  $26 billion               (MONEY)
    🔴 FP  1976                      (DATE)
    🔴 FP  2016                      (DATE)
    ✅ TP  Apple Inc.                (ORG)
    ✅ TP  Barack Obama              (PERSON)
    ✅ TP  California                (GPE)
    ✅ TP  Cupertino                 (GPE)
    ✅ TP  Elon Musk                 (PERSON)
    ✅ TP  LinkedIn                  (ORG)
    ✅ TP  Microsoft                 (ORG)
    ✅ TP  Steve Jobs                (PERSON)
    ✅ TP  Tesla                     (ORG)
    ⚪ FN  Washington D.C.           (GPE)
    🔴 FP  Washington D.C.           (ORG)


In [6]:
# ── 2b. Per-Entity-Type Metrics Breakdown ──────────────────
entity_types = set(e[1] for e in gold_entities + predicted_entities)

print('\n📊 Per-Entity-Type Metrics')
print('='*60)
print(f'  {"Type":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"TP":>5} {"FP":>5} {"FN":>5}')
print(f'  {"-"*57}')
for etype in sorted(entity_types):
    gold_e = set(e for e in gold_entities if e[1] == etype)
    pred_e = set(e for e in predicted_entities if e[1] == etype)
    tp = len(gold_e & pred_e)
    fp = len(pred_e - gold_e)
    fn = len(gold_e - pred_e)
    pr = tp/(tp+fp) if tp+fp else 0.0
    rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    print(f'  {etype:<12} {pr:>10.3f} {rc:>10.3f} {f1:>10.3f} {tp:>5} {fp:>5} {fn:>5}')


📊 Per-Entity-Type Metrics
  Type          Precision     Recall         F1    TP    FP    FN
  ---------------------------------------------------------
  DATE              0.000      0.000      0.000     0     2     0
  GPE               1.000      0.667      0.800     2     0     1
  MONEY             0.000      0.000      0.000     0     1     0
  ORG               0.800      1.000      0.889     4     1     0
  PERSON            1.000      1.000      1.000     3     0     0


In [7]:
# ── 2c. Exact Match vs Partial Match ──────────────────────
pred_vals = ['Steve Jobs', 'California', '1976', 'Tesla Motors', 'Barack']
gold_vals = ['Steve Jobs', 'California', '1976', 'Tesla',        'Barack Obama']

def exact_match(preds, golds):
    return sum(p.lower()==g.lower() for p,g in zip(preds,golds)) / len(golds)

def partial_match(preds, golds):
    scores = [len(set(p.lower().split()) & set(g.lower().split())) / len(set(g.lower().split()))
              for p,g in zip(preds,golds)]
    return float(np.mean(scores))

show('Exact vs Partial Match', {'Exact Match': exact_match(pred_vals, gold_vals),
                                 'Partial Match': partial_match(pred_vals, gold_vals)})
print('\n  Per-sample:')
for p, g in zip(pred_vals, gold_vals):
    icon = '✅' if p.lower()==g.lower() else '❌'
    print(f'  {icon}  pred: {p:<20}  gold: {g}')


  📊 Exact vs Partial Match
  Exact Match                   : 0.6000
  Partial Match                 : 0.9000

  Per-sample:
  ✅  pred: Steve Jobs            gold: Steve Jobs
  ✅  pred: California            gold: California
  ✅  pred: 1976                  gold: 1976
  ❌  pred: Tesla Motors          gold: Tesla
  ❌  pred: Barack                gold: Barack Obama


## 3. Edit Distance & Character-Level Metrics
### Tool: python-Levenshtein

In [8]:
import Levenshtein

# ── 3a. Levenshtein Distance ─────────────────────────────
extraction_pairs = [
    ("Steve Jobs",         "Steve Jobs"),       # perfect
    ("Elon Musk",          "Elon  Musk"),       # extra space
    ("Washington D.C.",    "Washington DC"),     # minor diff
    ("$26 billion",        "26 billion dollars"), # format diff
    ("Microsoft Corp",     "Microsoft"),         # partial
    ("Barack H. Obama",    "Barack Obama"),      # middle name
]

print('\n📏 Edit Distance & Character-Level Metrics')
print('='*70)
print(f'  {"Predicted":<25} {"Gold":<25} {"Lev Dist":>8} {"Ratio":>8} {"Jaro":>8} {"JaroW":>8}')
print(f'  {"-"*82}')
for pred, gold in extraction_pairs:
    lev_dist = Levenshtein.distance(pred, gold)
    lev_ratio = Levenshtein.ratio(pred, gold)
    jaro = Levenshtein.jaro(pred, gold)
    jaro_w = Levenshtein.jaro_winkler(pred, gold)
    print(f'  {pred:<25} {gold:<25} {lev_dist:>8} {lev_ratio:>8.3f} {jaro:>8.3f} {jaro_w:>8.3f}')

avg_ratio = np.mean([Levenshtein.ratio(p, g) for p, g in extraction_pairs])
avg_jaro = np.mean([Levenshtein.jaro_winkler(p, g) for p, g in extraction_pairs])
print(f'\n  Average Levenshtein Ratio: {avg_ratio:.4f}')
print(f'  Average Jaro-Winkler:     {avg_jaro:.4f}')


📏 Edit Distance & Character-Level Metrics
  Predicted                 Gold                      Lev Dist    Ratio     Jaro    JaroW
  ----------------------------------------------------------------------------------
  Steve Jobs                Steve Jobs                       0    1.000    1.000    1.000
  Elon Musk                 Elon  Musk                       1    0.947    0.967    0.980
  Washington D.C.           Washington DC                    2    0.929    0.956    0.973
  $26 billion               26 billion dollars               9    0.690    0.822    0.822
  Microsoft Corp            Microsoft                        5    0.783    0.881    0.929
  Barack H. Obama           Barack Obama                     3    0.889    0.933    0.960

  Average Levenshtein Ratio: 0.8728
  Average Jaro-Winkler:     0.9439


In [9]:
# ── 3b. Normalized Edit Distance & Character Error Rate ───
def char_error_rate(pred, gold):
    return Levenshtein.distance(pred, gold) / max(len(gold), 1)

def normalized_edit_distance(pred, gold):
    return Levenshtein.distance(pred, gold) / max(len(pred), len(gold), 1)

print('\n📏 Character Error Rate (CER) & Normalized Edit Distance')
print('='*60)
for pred, gold in extraction_pairs:
    cer = char_error_rate(pred, gold)
    ned = normalized_edit_distance(pred, gold)
    print(f'  CER={cer:.3f}  NED={ned:.3f}  |  "{pred}" vs "{gold}"')

avg_cer = np.mean([char_error_rate(p, g) for p, g in extraction_pairs])
print(f'\n  Average CER: {avg_cer:.4f}')


📏 Character Error Rate (CER) & Normalized Edit Distance
  CER=0.000  NED=0.000  |  "Steve Jobs" vs "Steve Jobs"
  CER=0.100  NED=0.100  |  "Elon Musk" vs "Elon  Musk"
  CER=0.154  NED=0.133  |  "Washington D.C." vs "Washington DC"
  CER=0.500  NED=0.500  |  "$26 billion" vs "26 billion dollars"
  CER=0.556  NED=0.357  |  "Microsoft Corp" vs "Microsoft"
  CER=0.250  NED=0.200  |  "Barack H. Obama" vs "Barack Obama"

  Average CER: 0.2599


## 4. LLM-Based Entity Extraction (Groq)
### Tool: Groq LLM for zero-shot NER + evaluation against gold standard

In [10]:
NER_SYS = '''You are a named entity recognition system. Extract ALL entities from the text.
Return JSON only: {"entities": [{"text": "...", "type": "PERSON|ORG|GPE|DATE|MONEY|PRODUCT"}]}'''

test_texts = [
    {
        "text": "Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976.",
        "gold": [("Apple Inc.", "ORG"), ("Steve Jobs", "PERSON"), ("Cupertino", "GPE"), ("California", "GPE"), ("1976", "DATE")]
    },
    {
        "text": "Tesla CEO Elon Musk announced a $44 billion deal to acquire Twitter in April 2022.",
        "gold": [("Tesla", "ORG"), ("Elon Musk", "PERSON"), ("$44 billion", "MONEY"), ("Twitter", "ORG"), ("April 2022", "DATE")]
    },
    {
        "text": "Google DeepMind, based in London, developed AlphaFold to predict protein structures.",
        "gold": [("Google DeepMind", "ORG"), ("London", "GPE"), ("AlphaFold", "PRODUCT")]
    },
]

print('\n🤖 LLM-Based NER Extraction + Evaluation')
print('='*70)
all_tp, all_fp, all_fn = 0, 0, 0

for test in test_texts:
    result = parse_json(groq_chat(test["text"], system=NER_SYS))
    pred_entities = set()
    for e in result.get("entities", []):
        pred_entities.add((e.get("text", ""), e.get("type", "")))
    
    gold_set = set(test["gold"])
    # Flexible matching: match on text only (type may vary)
    gold_texts = {g[0].lower() for g in gold_set}
    pred_texts = {p[0].lower() for p in pred_entities}
    
    tp = len(gold_texts & pred_texts)
    fp = len(pred_texts - gold_texts)
    fn = len(gold_texts - pred_texts)
    all_tp += tp; all_fp += fp; all_fn += fn
    
    pr = tp/(tp+fp) if tp+fp else 0.0
    rc = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*pr*rc/(pr+rc) if pr+rc else 0.0
    
    print(f'\n  Text: {test["text"][:70]}...')
    print(f'  Gold:      {[g[0] for g in test["gold"]]}')
    print(f'  Predicted: {[p[0] for p in pred_entities]}')
    print(f'  P={pr:.2f}  R={rc:.2f}  F1={f1:.2f}  (TP={tp}, FP={fp}, FN={fn})')

total_pr = all_tp/(all_tp+all_fp) if all_tp+all_fp else 0
total_rc = all_tp/(all_tp+all_fn) if all_tp+all_fn else 0
total_f1 = 2*total_pr*total_rc/(total_pr+total_rc) if total_pr+total_rc else 0
print(f'\n  Overall Micro-averaged: P={total_pr:.3f}  R={total_rc:.3f}  F1={total_f1:.3f}')


🤖 LLM-Based NER Extraction + Evaluation

  Text: Apple Inc. was founded by Steve Jobs in Cupertino, California in 1976....
  Gold:      ['Apple Inc.', 'Steve Jobs', 'Cupertino', 'California', '1976']
  Predicted: ['Apple Inc.', 'Steve Jobs', 'Cupertino', 'California', '1976']
  P=1.00  R=1.00  F1=1.00  (TP=5, FP=0, FN=0)

  Text: Tesla CEO Elon Musk announced a $44 billion deal to acquire Twitter in...
  Gold:      ['Tesla', 'Elon Musk', '$44 billion', 'Twitter', 'April 2022']
  Predicted: ['Tesla', '$44 billion', 'April 2022', 'Elon Musk', 'Twitter']
  P=1.00  R=1.00  F1=1.00  (TP=5, FP=0, FN=0)

  Text: Google DeepMind, based in London, developed AlphaFold to predict prote...
  Gold:      ['Google DeepMind', 'London', 'AlphaFold']
  Predicted: ['AlphaFold', 'London', 'Google DeepMind']
  P=1.00  R=1.00  F1=1.00  (TP=3, FP=0, FN=0)

  Overall Micro-averaged: P=1.000  R=1.000  F1=1.000


## 5. Relation Extraction Evaluation
### Tool: Groq LLM for zero-shot relation extraction

In [11]:
RE_SYS = '''You are a relation extraction system. Extract relationships between entities.
Return JSON only: {"relations": [{"subject": "...", "relation": "...", "object": "..."}]}'''

re_tests = [
    {
        "text": "Steve Jobs co-founded Apple Inc. in Cupertino, California. Tim Cook succeeded him as CEO in 2011.",
        "gold_relations": [
            ("Steve Jobs", "co-founded", "Apple Inc."),
            ("Apple Inc.", "located_in", "Cupertino"),
            ("Tim Cook", "succeeded", "Steve Jobs"),
            ("Tim Cook", "CEO_of", "Apple Inc."),
        ]
    },
    {
        "text": "Elon Musk is the CEO of Tesla and SpaceX. Tesla is headquartered in Austin, Texas.",
        "gold_relations": [
            ("Elon Musk", "CEO_of", "Tesla"),
            ("Elon Musk", "CEO_of", "SpaceX"),
            ("Tesla", "headquartered_in", "Austin"),
        ]
    },
]

print('\n🔗 Relation Extraction Evaluation')
print('='*70)
total_found, total_gold = 0, 0

for test in re_tests:
    result = parse_json(groq_chat(test["text"], system=RE_SYS, max_tokens=400))
    pred_rels = result.get("relations", [])
    
    print(f'\n  Text: {test["text"][:70]}...')
    print(f'  Gold relations ({len(test["gold_relations"])}):')
    for s, r, o in test["gold_relations"]:
        print(f'    ({s}) --[{r}]--> ({o})')
    
    print(f'  Predicted relations ({len(pred_rels)}):')
    found = 0
    for rel in pred_rels:
        s, r, o = rel.get("subject",""), rel.get("relation",""), rel.get("object","")
        # Fuzzy match: check if subject and object appear in any gold triple
        match = any(
            s.lower() in gs.lower() or gs.lower() in s.lower()
            for gs, gr, go in test["gold_relations"]
            if o.lower() in go.lower() or go.lower() in o.lower()
        )
        icon = "✅" if match else "⚠️"
        if match: found += 1
        print(f'    {icon} ({s}) --[{r}]--> ({o})')
    
    total_found += found
    total_gold += len(test["gold_relations"])
    recall = found / len(test["gold_relations"])
    print(f'  Recall: {recall:.1%} ({found}/{len(test["gold_relations"])})')

print(f'\n  Overall Relation Recall: {total_found}/{total_gold} = {total_found/total_gold:.1%}')


🔗 Relation Extraction Evaluation

  Text: Steve Jobs co-founded Apple Inc. in Cupertino, California. Tim Cook su...
  Gold relations (4):
    (Steve Jobs) --[co-founded]--> (Apple Inc.)
    (Apple Inc.) --[located_in]--> (Cupertino)
    (Tim Cook) --[succeeded]--> (Steve Jobs)
    (Tim Cook) --[CEO_of]--> (Apple Inc.)
  Predicted relations (4):
    ✅ (Steve Jobs) --[co-founded]--> (Apple Inc.)
    ✅ (Tim Cook) --[succeeded]--> (Steve Jobs)
    ✅ (Apple Inc.) --[located in]--> (Cupertino, California)
    ✅ (Tim Cook) --[became CEO of]--> (Apple Inc.)
  Recall: 100.0% (4/4)

  Text: Elon Musk is the CEO of Tesla and SpaceX. Tesla is headquartered in Au...
  Gold relations (3):
    (Elon Musk) --[CEO_of]--> (Tesla)
    (Elon Musk) --[CEO_of]--> (SpaceX)
    (Tesla) --[headquartered_in]--> (Austin)
  Predicted relations (3):
    ✅ (Elon Musk) --[is CEO of]--> (Tesla)
    ✅ (Elon Musk) --[is CEO of]--> (SpaceX)
    ✅ (Tesla) --[is headquartered in]--> (Austin, Texas)
  Recall: 100.0% (3/3)

## 6. Schema Conformity (Pydantic + jsonschema + JSON validity)
### Tool: Pydantic validation + jsonschema

In [12]:
from pydantic import BaseModel, Field, ValidationError, field_validator
from typing import Optional, List
import jsonschema

# ── 6a. Pydantic Schema Validation ───────────────────────
class ExtractedPerson(BaseModel):
    name:       str           = Field(..., min_length=2)
    age:        Optional[int] = Field(None, ge=0, le=150)
    occupation: Optional[str] = None
    skills:     List[str]     = Field(default_factory=list)

    @field_validator('name')
    @classmethod
    def must_have_upper(cls, v):
        if not any(c.isupper() for c in v):
            raise ValueError('Name must contain at least one capital letter')
        return v

llm_outputs = [
    '{"name": "Alice Johnson", "age": 32, "occupation": "Engineer", "skills": ["Python","ML"]}',
    '{"name": "bob", "age": 25}',
    '{"name": "Carol", "age": 200}',
    '{"name": "D"}',
    '{"name": "Eve Chen", "skills": []}',
    'not json at all',
    '{"name": "Frank Lee", "age": -5}',
    '{"name": "Grace Hopper", "age": 85, "occupation": "Computer Scientist", "skills": ["COBOL", "Compilers"]}',
]

print('\n📋 Pydantic Schema Validation')
print('='*60)
valid = 0
errors_by_type = {"json_error": 0, "validation_error": 0}
for i, raw in enumerate(llm_outputs):
    try:
        p = ExtractedPerson(**json.loads(raw))
        print(f'  [{i}] ✅ VALID    {p.model_dump()}')
        valid += 1
    except json.JSONDecodeError as e:
        errors_by_type["json_error"] += 1
        print(f'  [{i}] ❌ JSON ERR  {e}')
    except ValidationError as e:
        errors_by_type["validation_error"] += 1
        errs = [f"{err['loc'][0]}: {err['msg']}" for err in e.errors()]
        print(f'  [{i}] ❌ SCHEMA   {errs}')
print(f'\n  Schema Conformity Rate: {valid}/{len(llm_outputs)} = {valid/len(llm_outputs):.1%}')
print(f'  JSON Parse Errors:     {errors_by_type["json_error"]}')
print(f'  Validation Errors:     {errors_by_type["validation_error"]}')


📋 Pydantic Schema Validation
  [0] ✅ VALID    {'name': 'Alice Johnson', 'age': 32, 'occupation': 'Engineer', 'skills': ['Python', 'ML']}
  [1] ❌ SCHEMA   ['name: Value error, Name must contain at least one capital letter']
  [2] ❌ SCHEMA   ['age: Input should be less than or equal to 150']
  [3] ❌ SCHEMA   ['name: String should have at least 2 characters']
  [4] ✅ VALID    {'name': 'Eve Chen', 'age': None, 'occupation': None, 'skills': []}
  [5] ❌ JSON ERR  Expecting value: line 1 column 1 (char 0)
  [6] ❌ SCHEMA   ['age: Input should be greater than or equal to 0']
  [7] ✅ VALID    {'name': 'Grace Hopper', 'age': 85, 'occupation': 'Computer Scientist', 'skills': ['COBOL', 'Compilers']}

  Schema Conformity Rate: 3/8 = 37.5%
  JSON Parse Errors:     1
  Validation Errors:     4


In [13]:
# ── 6b. JSON Schema Validation ────────────────────────────
schema = {
    'type': 'object',
    'required': ['sql', 'tables_used', 'confidence'],
    'properties': {
        'sql':         {'type': 'string'},
        'tables_used': {'type': 'array', 'items': {'type': 'string'}},
        'confidence':  {'type': 'number', 'minimum': 0, 'maximum': 1},
    },
    'additionalProperties': False
}
json_tests = [
    {'sql': 'SELECT * FROM users', 'tables_used': ['users'], 'confidence': 0.9},
    {'sql': 'SELECT id',           'tables_used': 'orders',  'confidence': 0.8},  # wrong type
    {'sql': 'SELECT *',            'confidence': 1.5},                             # missing field + out of range
    {'sql': 'SELECT name', 'tables_used': ['users'], 'confidence': 0.7, 'extra': 'field'},  # additional prop
    {'sql': 'SELECT *', 'tables_used': [], 'confidence': 0.5},                    # empty array (valid)
]
print('\n📋 JSON Schema Validation')
print('='*60)
valid_count = 0
for i, obj in enumerate(json_tests):
    try:
        jsonschema.validate(obj, schema)
        print(f'  [{i}] ✅ VALID   {json.dumps(obj)[:60]}')
        valid_count += 1
    except jsonschema.ValidationError as e:
        print(f'  [{i}] ❌ INVALID  {e.message[:70]}')
print(f'\n  JSON Schema Conformity: {valid_count}/{len(json_tests)} = {valid_count/len(json_tests):.1%}')


📋 JSON Schema Validation
  [0] ✅ VALID   {"sql": "SELECT * FROM users", "tables_used": ["users"], "co
  [1] ❌ INVALID  'orders' is not of type 'array'
  [2] ❌ INVALID  'tables_used' is a required property
  [3] ❌ INVALID  Additional properties are not allowed ('extra' was unexpected)
  [4] ✅ VALID   {"sql": "SELECT *", "tables_used": [], "confidence": 0.5}

  JSON Schema Conformity: 2/5 = 40.0%


## 7. Table / Structured Data Extraction Evaluation
### Tool: Groq LLM for extracting tabular data

In [14]:
TABLE_SYS = '''Extract the table data from the text into structured JSON.
Return JSON only: {"headers": ["col1", "col2", ...], "rows": [["val1", "val2", ...], ...]}'''

table_texts = [
    {
        "text": "Product prices: Laptop costs $999, Mouse costs $29, Keyboard costs $79, and Monitor costs $399.",
        "gold": {"headers": ["Product", "Price"], "rows": [["Laptop","$999"],["Mouse","$29"],["Keyboard","$79"],["Monitor","$399"]]}
    },
    {
        "text": "Employee directory: John Smith is in Engineering, age 35. Jane Doe is in Marketing, age 28. Bob Wilson is in Sales, age 42.",
        "gold": {"headers": ["Name", "Department", "Age"], "rows": [["John Smith","Engineering","35"],["Jane Doe","Marketing","28"],["Bob Wilson","Sales","42"]]}
    },
]

print('\n📊 Table Extraction Evaluation')
print('='*65)
for test in table_texts:
    result = parse_json(groq_chat(test["text"], system=TABLE_SYS, max_tokens=400))
    gold = test["gold"]
    
    pred_rows = result.get("rows", [])
    gold_rows = gold["rows"]
    
    # Row-level match
    matched_rows = 0
    for grow in gold_rows:
        for prow in pred_rows:
            # Check if values overlap significantly
            grow_lower = [str(v).lower().replace("$","").strip() for v in grow]
            prow_lower = [str(v).lower().replace("$","").strip() for v in prow]
            overlap = len(set(grow_lower) & set(prow_lower))
            if overlap >= len(grow) * 0.5:
                matched_rows += 1
                break
    
    row_recall = matched_rows / len(gold_rows)
    col_match = len(result.get("headers", [])) == len(gold["headers"])
    
    print(f'\n  Text: {test["text"][:70]}...')
    print(f'  Gold: {len(gold_rows)} rows, {len(gold["headers"])} cols')
    print(f'  Pred: {len(pred_rows)} rows, {len(result.get("headers",[]))} cols')
    print(f'  Row Recall: {row_recall:.1%}  Column Match: {"✅" if col_match else "❌"}')


📊 Table Extraction Evaluation

  Text: Product prices: Laptop costs $999, Mouse costs $29, Keyboard costs $79...
  Gold: 4 rows, 2 cols
  Pred: 4 rows, 2 cols
  Row Recall: 100.0%  Column Match: ✅

  Text: Employee directory: John Smith is in Engineering, age 35. Jane Doe is ...
  Gold: 3 rows, 3 cols
  Pred: 3 rows, 3 cols
  Row Recall: 100.0%  Column Match: ✅


## 8. Aggregated Text Extraction Scorecard

In [15]:
print('\n' + '='*60)
print('  📊 TEXT EXTRACTION EVALUATION SCORECARD')
print('='*60)
print(f'''
  Metric Category                  Score
  ───────────────────────────────  ──────
  NER Precision/Recall/F1          See Section 2a
  Per-Entity-Type Breakdown        See Section 2b
  Exact Match Rate                 {exact_match(pred_vals, gold_vals):.1%}
  Partial Match Rate               {partial_match(pred_vals, gold_vals):.1%}
  Avg Levenshtein Ratio            {avg_ratio:.4f}
  Avg Jaro-Winkler                 {avg_jaro:.4f}
  Avg Character Error Rate         {avg_cer:.4f}
  LLM NER Micro-F1                 {total_f1:.3f}
  Relation Recall                  {total_found}/{total_gold}
  Schema Conformity (Pydantic)     {valid}/{len(llm_outputs)}
  Schema Conformity (jsonschema)   {valid_count}/{len(json_tests)}
''')


  📊 TEXT EXTRACTION EVALUATION SCORECARD

  Metric Category                  Score
  ───────────────────────────────  ──────
  NER Precision/Recall/F1          See Section 2a
  Per-Entity-Type Breakdown        See Section 2b
  Exact Match Rate                 60.0%
  Partial Match Rate               90.0%
  Avg Levenshtein Ratio            0.8728
  Avg Jaro-Winkler                 0.9439
  Avg Character Error Rate         0.2599
  LLM NER Micro-F1                 1.000
  Relation Recall                  7/7
  Schema Conformity (Pydantic)     3/8
  Schema Conformity (jsonschema)   2/5

